In [ ]:
!pip install transformers seqeval evaluate datasets

### Data Loading

In [ ]:
from datasets import load_dataset
dataset_raw= load_dataset("lfcc/portuguese_ner")
dataset_raw


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [ ]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

### Data Pre-Processing

In [ ]:
from transformers import AutoTokenizer
#usar o tokanaizer usado para treinar o modelo que tamos a usar ?
tokenizer=AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [ ]:
inputs=tokenizer("As aulas de PLN são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)
#cls trieno do modelo, classifica se as frases fazem sentido ou não
#sep serpara as frases

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [ ]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

In [ ]:
tokens=["as","aulas","plneb","são","interessantes","!"]
inputs=tokenizer(tokens, is_split_into_words=True)
newtokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(newtokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [ ]:
len(tokens), len(newtokens)

(6, 10)

In [ ]:
inputs.word_ids() #mapeamento entre tokens antigos e tokens novos

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [ ]:
print(dataset_raw)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})


In [ ]:
def align_labels_with_tokens(word_ids,labels):
    new_labels=[]
    previous_word= None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)#codigo para igonorar a label
        elif previous_word != word_id:
            new_labels.append(labels[word_id])#mantem a label
        else:
            new_labels.append(-100)#ignorar as sub words
        previous_word= word_id
    return new_labels

def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(row["tokens"], is_split_into_words=True, truncation=True,max_length=512)
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels[:len(inputs["input_ids"])]
        res.append(inputs)
    return res

train_data= tokenize_dataset(dataset_raw["train"])
test_data= tokenize_dataset(dataset_raw["test"])
len(train_data), len(test_data)



(3716, 930)

In [ ]:
from datasets import Dataset
train_dataset=Dataset.from_list(train_data)
test_dataset=Dataset.from_list(test_data)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


In [ ]:
from transformers import AutoModelForTokenClassification

model= AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [ ]:
label_list = dataset_raw["train"].features["ner_tags"].feature.names
label_list

['O',
 'B-Data',
 'I-Data',
 'B-Local',
 'I-Local',
 'B-Organizacao',
 'I-Organizacao',
 'B-Pessoa',
 'I-Pessoa',
 'B-Profissao',
 'I-Profissao']

Evaluation

In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

In [ ]:
import numpy as np

#labels = [label_list[i] for i in example[f"ner_tags"]]


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

In [ ]:
id2label={}
i=0
for label in label_list:
  id2label[i]=label
  i=i+1
label2id={v:k for k, v in id2label.items()}
print(id2label)
print(label2id)

{0: 'O', 1: 'B-Data', 2: 'I-Data', 3: 'B-Local', 4: 'I-Local', 5: 'B-Organizacao', 6: 'I-Organizacao', 7: 'B-Pessoa', 8: 'I-Pessoa', 9: 'B-Profissao', 10: 'I-Profissao'}
{'O': 0, 'B-Data': 1, 'I-Data': 2, 'B-Local': 3, 'I-Local': 4, 'B-Organizacao': 5, 'I-Organizacao': 6, 'B-Pessoa': 7, 'I-Pessoa': 8, 'B-Profissao': 9, 'I-Profissao': 10}


In [ ]:

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir="modelo_pt_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    #push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.073528,0.929707,0.962963,0.946043,0.982223
2,No log,0.067353,0.950794,0.966445,0.958556,0.985025
3,0.127103,0.065778,0.947254,0.966445,0.956753,0.984456
4,0.127103,0.067281,0.947270,0.966762,0.956917,0.984894
5,0.024920,0.070462,0.947270,0.966762,0.956917,0.984850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1165, training_loss=0.06760655370393025, metrics={'train_runtime': 580.8567, 'train_samples_per_second': 31.987, 'train_steps_per_second': 2.006, 'total_flos': 1094742839483712.0, 'train_loss': 0.06760655370393025, 'epoch': 5.0})

In [ ]:
texto_noticia="""O presidente norte-americano, Donald Trump, garantiu hoje que Washington tem tido negociações "muito boas" com Teerão nas últimas horas e considerou "muito possível" um acordo para pôr fim à guerra contra o Irão que fechou o estreito de Ormuz."""

In [ ]:
from transformers import pipeline

classifier = pipeline("ner",model=model, tokenizer=tokenizer, aggregation_strategy="first")
classifier(texto_noticia)

[{'entity_group': 'Profissao',
  'score': np.float32(0.5663152),
  'word': 'presidente',
  'start': 2,
  'end': 12},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.60571367),
  'word': 'Donald Trump',
  'start': 30,
  'end': 42},
 {'entity_group': 'Local',
  'score': np.float32(0.8932498),
  'word': 'Washington',
  'start': 62,
  'end': 72},
 {'entity_group': 'Local',
  'score': np.float32(0.955549),
  'word': 'Teerão',
  'start': 111,
  'end': 117},
 {'entity_group': 'Local',
  'score': np.float32(0.819712),
  'word': 'Irão',
  'start': 207,
  'end': 211},
 {'entity_group': 'Local',
  'score': np.float32(0.92741996),
  'word': 'Ormuz',
  'start': 237,
  'end': 242}]